# Query on Databricks (serverless) via Databricks Connect

**Kernel:** select the `connect/.venv` interpreter (top right > Select Kernel > Python Environments > `.venv` from this `connect` folder).

Uses the `community` profile (workspace dbc-31b4ebc3-548d) and serverless compute. The first query may take a few seconds due to serverless cold start.

In [ ]:
from databricks.connect import DatabricksSession
from databricks.sdk.core import Config
import pyspark.sql.functions as F

# Forced auth_type="pat": avoids relying on the local credential broker from
# the VS Code Databricks extension (auth_type=metadata-service) -- if that broker
# stops or changes port, the session fails with ConnectionRefusedError even with
# profile="community" configured, because env vars injected by the extension take
# precedence over the profile. Forcing PAT here ignores those env vars.
#
# .serverless(True) on the builder counts as an "explicit connection parameter" and
# cannot be combined with sdkConfig() -- so serverless_compute_id is set
# directly on Config without calling .serverless() on the builder.
cfg = Config(profile="community", auth_type="pat")
cfg.serverless_compute_id = "auto"
spark = DatabricksSession.builder.sdkConfig(cfg).getOrCreate()
spark

In [ ]:
# Test query
df = spark.sql("SELECT 1 AS ok, current_catalog() AS catalog, current_user() AS user")
df.show()

In [ ]:
df = spark.read.parquet("/Volumes/example/raw/landing_data/customer/")
# df = df.filter(df.productcategoryid.isNotNull())
# df.select("persontype").distinct()
df.show(5, truncate= False)

In [ ]:
spark.sql(""" use catalog advworks_prd """)

In [ ]:
%sql
select count(*) from advworks_prd.trusted.customer;
